## Train the first model pipeline

### By:
[Gabriel Múnera](https://github.com/gamug)

### Date:
2026-08-21

### Description:

Set all the code needed to train, tune and persist the first production model for diamond `price` prediction,
selected in `notebooks/5-models/02-gmg-basic_algorithms_model_selection-2026_08_18.ipynb`:
**`HistGradientBoostingRegressor`**, wrapped in a `TransformedTargetRegressor` that models `log(price)`, on top
of the preprocessing pipeline designed in
`notebooks/4-feat_eng/01-gmg-basic-feature-engineering-pipeline-2026_08_18.ipynb`.

This notebook is meant to be a clean, self-contained, re-runnable script version of the winning experiment: load
data → prepare → feature engineer → split → tune → evaluate → **save the final model artifact to
`data/06_models/`**.

## 📚 Import  libraries

In [1]:
# base libraries for data science
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, StandardScaler

## 💾 Load data

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

dataset = pd.read_parquet(DATA_DIR / "04_feature/diamantes_clean.parquet", engine="pyarrow")
dataset.shape

(18977, 10)

## 👷 Data preparation

The columns that will be used are:

`['carat', 'depth', 'table', 'x', 'y', 'z', 'cut', 'color', 'clarity']` to predict `price`.

`data/04_feature/diamantes_clean.parquet` is already deduplicated and free of the corrupted rows identified in
the EDA (`z == carat`, `y == depth` data-entry swaps) — that cleaning happened upstream in the
feature-engineering pipeline notebook, so it is not repeated here.

In [ ]:
numeric_features = ["carat", "depth", "table"]
volume_source_features = ["x", "y", "z"]
categorical_ordinal_features = ["cut", "color", "clarity"]
target = "price"

selected_features = numeric_features + volume_source_features + categorical_ordinal_features
dataset_features = dataset[[*selected_features, target]]
dataset_features.isna().sum()

carat      0
depth      0
table      0
x          0
y          0
z          0
cut        0
color      0
clarity    0
price      0
dtype: int64

## Convert data types

In [4]:
# numerical columns are already float32 from the upstream pipeline; categorical grades are pandas `category`
dataset_features[numeric_features + volume_source_features] = dataset_features[
    numeric_features + volume_source_features
].astype("float32")

dataset_features[categorical_ordinal_features] = dataset_features[
    categorical_ordinal_features
].astype("category")
dataset_features[target] = dataset_features[target].astype("int64")
dataset_features.dtypes

carat       float32
depth       float32
table       float32
x           float32
y           float32
z           float32
cut        category
color      category
clarity    category
price         int64
dtype: object

## 👨‍🏭 Feature Engineering

Same `ColumnTransformer` validated in the feature-engineering and model-selection notebooks: `volume = x*y*z`
replaces the raw dimensions (multicollinearity fix), and `cut`/`color`/`clarity` are ordinal-encoded worst → best.

In [ ]:
ordinal_categories = {
    "cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    "color": ["J", "I", "H", "G", "F", "E", "D"],
    "clarity": ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
}


def compute_volume(X: np.ndarray) -> np.ndarray:
    x, y, z = X[:, 0], X[:, 1], X[:, 2]
    return (x * y * z).reshape(-1, 1)


def volume_feature_names_out(
    transformer: FunctionTransformer, input_features: list[str]
) -> np.ndarray:
    """Named (picklable) replacement for a lambda: FunctionTransformer always outputs 1 column, 'volume'."""
    return np.array(["volume"])


volume_transformer = FunctionTransformer(compute_volume, feature_names_out=volume_feature_names_out)

numeric_pipe = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
volume_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("volume", volume_transformer),
        ("scaler", StandardScaler()),
    ]
)
categorical_ord_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=[ordinal_categories[c] for c in categorical_ordinal_features]
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("volume", volume_pipe, volume_source_features),
        ("cat_ordinal", categorical_ord_pipe, categorical_ordinal_features),
    ]
)

## Train / Test split

In [6]:
# split data into features and target

X_features = dataset_features[selected_features]
Y_target = dataset_features[target]

# stratify on clarity x cut, as recommended in the feature-engineering notebook: some grade combinations
# are rare, so a plain random split risks under/over-representing them between train and test.
clarity_cut_key = (
    dataset_features["clarity"].astype(str) + "_" + dataset_features["cut"].astype(str)
)

x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, stratify=clarity_cut_key, test_size=0.2, random_state=42
)
x_train.shape, x_test.shape

((15181, 9), (3796, 9))

### Create pipeline

In [7]:
# log-price target transform, per the feature-engineering notebook's recommendation: linearizes the convex
# carat -> price relationship and guarantees strictly positive predictions.
data_model_pipeline = TransformedTargetRegressor(
    regressor=Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", HistGradientBoostingRegressor(random_state=42)),
        ]
    ),
    func=np.log,
    inverse_func=np.exp,
)

## Hyperparameter tunning

Select the best hyperparameters for `HistGradientBoostingRegressor`, the model selected in
`02-...model_selection.ipynb` (best cross-validated MAPE / R² among all candidates, including the tuned
`GradientBoostingRegressor` runner-up).

### Hist Gradient Boosting

In [8]:
score = "neg_mean_absolute_percentage_error"

hyperparameters = {
    "regressor__model__max_iter": [200, 300, 400],
    "regressor__model__max_depth": [6, 10, None],
    "regressor__model__learning_rate": [0.03, 0.05, 0.1],
    "regressor__model__l2_regularization": [0.0, 1.0],
}

grid_search = GridSearchCV(
    data_model_pipeline,
    hyperparameters,
    cv=5,
    scoring=score,
    n_jobs=-1,
)
grid_search.fit(x_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",TransformedTa..._state=42))]))
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__model__l2_regularization': [0.0, 1.0], 'regressor__model__learning_rate': [0.03, 0.05, ...], 'regressor__model__max_depth': [6, 10, ...], 'regressor__model__max_iter': [200, 300, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_percentage_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance mod

In [9]:
grid_search.best_params_

{'regressor__model__l2_regularization': 0.0,
 'regressor__model__learning_rate': 0.05,
 'regressor__model__max_depth': 10,
 'regressor__model__max_iter': 300}

In [10]:
best_data_model_pipeline = grid_search.best_estimator_
best_data_model_pipeline

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'exp'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](9,)","['carat','depth','table',...,'cut','color','clarity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,9
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...m_state=42))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"


### Evaluation

In [11]:
def regression_metrics(y_true, y_pred) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAPE_%": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "R2": r2_score(y_true, y_pred),
    }


y_pred = best_data_model_pipeline.predict(x_test)
test_metrics = regression_metrics(y_test, y_pred)
test_metrics

{'MAE': 559.3547288047087,
 'RMSE': 856.3277810659189,
 'MAPE_%': 6.8373308711976515,
 'R2': 0.9517919914168572}

In [12]:
# compare against the CV score used during tuning
cv_mape = -grid_search.best_score_ * 100
print(f"Best CV MAPE: {cv_mape:.2f}%")
print(f"Test MAPE:    {test_metrics['MAPE_%']:.2f}%")

Best CV MAPE: 7.10%
Test MAPE:    6.84%


## Save the model

In [13]:
MODEL_DIR = DATA_DIR / "06_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR

PosixPath('/workspaces/diamantes/data/06_models')

In [14]:
# save the fitted model pipeline (preprocessing + log-target transform + HistGradientBoostingRegressor)
dump(
    best_data_model_pipeline,
    MODEL_DIR / "diamantes_price-hist_gradient_boosting-v1.joblib",
    protocol=5,
)

['/workspaces/diamantes/data/06_models/diamantes_price-hist_gradient_boosting-v1.joblib']

#### Test the saved model

In [15]:
from joblib import load

my_model = load(MODEL_DIR / "diamantes_price-hist_gradient_boosting-v1.joblib")

In [16]:
# check predictions
x_test.head()

,carat,depth,table,x,y,z,cut,color,clarity
15363,1.51,63.000000,58.0,7.33,7.27,4.60,Premium,E,SI1
13103,1.20,61.400002,55.0,6.78,6.87,4.19,Ideal,D,VS2
13678,1.80,63.200001,58.0,7.66,7.75,4.87,Good,I,SI2
16426,1.73,61.000000,56.0,7.72,7.79,4.73,Ideal,I,VS2
5308,1.00,62.000000,55.0,6.39,6.44,3.98,Ideal,F,SI1


In [17]:
my_model.predict(x_test.head())

array([11384.74059944,  9663.30063233,  9255.00467717, 11940.54146152,
        5294.05475894])

In [18]:
y_test.head().to_numpy()

array([11873,  9424,  9979, 13465,  5239])

## 📊 Analysis of Results

- **Best hyperparameters**: `max_iter=300, max_depth=10, learning_rate=0.05, l2_regularization=0.0` — the same
  optimum independently found in the wider comparison of `02-...model_selection.ipynb`, which is a good sign the
  search grid was well-centered.
- **Test performance**: MAE ≈ \$559.35, RMSE ≈ \$856.33, **MAPE ≈ 6.84%**, **R² ≈ 0.952**.
- **CV MAPE (7.10%) vs test MAPE (6.84%)** are close, with test slightly better than the CV estimate — no sign
  of overfitting to the training folds, and the held-out test set was reserved from tuning entirely.
- **Sanity check**: reloading the saved `.joblib` pipeline and predicting on 5 held-out rows shows predictions
  within roughly 4-15% of the true price for each example (e.g. \$11,384.74 predicted vs \$11,873 actual;
  \$5,294.05 vs \$5,239 actual) — consistent with the ~6.8% average MAPE and confirms the persisted artifact
  round-trips correctly (predict-after-load matches the in-memory model).
- This is a **large improvement over the heuristic baseline** (MAPE ≈ 12.5%, R² ≈ 0.84 in
  `01-gmg-base_model-2026_08_18.ipynb`), at the cost of interpretability — the model is now a black-box
  ensemble instead of readable rate tables.

_(filled in after execution)_

## 🧑‍🔬 Recommendations:

1. **This is the model to promote as the project's first production candidate.** It is saved at
   `data/06_models/diamantes_price-hist_gradient_boosting-v1.joblib` as a complete, self-contained
   `TransformedTargetRegressor` pipeline (imputation + scaling + ordinal encoding + volume engineering + the
   tuned regressor all bundled together), so downstream code only needs `joblib.load(...).predict(new_data)`
   with a raw `carat, depth, table, x, y, z, cut, color, clarity` DataFrame — no separate preprocessing step to
   keep in sync.
2. **Track this run with MLflow** (see `04-...experiment-track-model.ipynb`) so the hyperparameters, metrics and
   model artifact are versioned and comparable against future retrains.
3. **Run the Deepchecks validation suite** (see `05-...deepcheck-ML-process.ipynb`) before treating this as
   truly production-ready — train/test drift, label leakage and weak-segment checks have not been run yet at
   this point in the pipeline.
4. **Scope caveat carried over from the feature-engineering notebook**: the training data top-codes at
   `carat ≈ 1.00`'s neighborhood distribution-wise (few very large stones); predictions for exceptionally large
   or unusually graded diamonds outside the training distribution should be treated with reduced confidence.
5. **Re-tune periodically as more labeled sales data becomes available** — the learning curve in
   `02-...model_selection.ipynb` showed validation MAPE was still improving at the largest available training
   size, so this is very likely not yet the accuracy ceiling for this model family.

_(filled in after execution)_

## 📖 References

- EDA notebook: `notebooks/3-analysis/02-jrz-data_description_Manual-pandas-2024_10_24.ipynb`
- Feature engineering notebook: `notebooks/4-feat_eng/01-gmg-basic-feature-engineering-pipeline-2026_08_18.ipynb`
- Model selection notebook: `notebooks/5-models/02-gmg-basic_algorithms_model_selection-2026_08_18.ipynb`
- <https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting>
- <https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html>